# 05. MAPA DE CALOR DE ACIDENTES POR VIA

CONTA QUANTOS ACIDENTES CADA VIA RECEBEU E DESENHA A MALHA COLORIDA POR ESSA CONTAGEM:
QUANTO MAIS ACIDENTES, MAIS VERMELHA A VIA.

DEPENDE DE DOIS NOTEBOOKS:

- **02**, QUE CRIA `vias_processadas` COM A GEOMETRIA E O `via_id` DE CADA VIA.
- **04**, QUE PREENCHE `via_id_associada` EM `acidentes_revisado`.

## O QUE SAI DAQUI

1. A TABELA `acidentes_por_via` NO BANCO, UMA LINHA POR VIA COM SUA CONTAGEM.
2. UM ARQUIVO HTML COM O MAPA, PARA ABRIR NO NAVEGADOR.

## POR QUE A COR VEM DA LINHA, E NAO DE UM BORRAO

O `HeatMap` DO FOLIUM SO ACEITA **PONTOS**. AQUI A UNIDADE DE ANALISE E A VIA INTEIRA,
QUE E UMA LINHA -- ENTAO O QUE COLORIMOS E A PROPRIA GEOMETRIA DA VIA.

ISSO E MAIS HONESTO QUE UM BORRAO: A CONTAGEM PERTENCE A VIA, NAO A UM PONTO DELA. NAO
SABEMOS **ONDE** NA VIA O ACIDENTE ACONTECEU, E UM MAPA DE PONTOS FINGIRIA QUE SABEMOS.

## ANTES DE ACREDITAR NO MAPA

O `via_id` E DERIVADO DA GEOMETRIA. SE VOCE REPROCESSOU O NOTEBOOK 02 DEPOIS DE RODAR O
04, OS IDENTIFICADORES MUDARAM E OS VINCULOS APONTAM PARA VIAS QUE NAO EXISTEM MAIS.

A ETAPA 2 MEDE EXATAMENTE ISSO E **AVISA**. SE O NUMERO DE ORFAOS VIER ALTO, O MAPA ESTA
MOSTRANDO MENOS ACIDENTES DO QUE VOCE TEM: RODE O NOTEBOOK 04 DE NOVO ANTES DE OLHAR.

## 1. IMPORTACAO DAS BIBLIOTECAS E CONFIGURACOES

EXPLICACAO DOS VALORES:

- `CODIGO_IBGE`: O MUNICIPIO DO MAPA. TEM QUE SER O MESMO QUE ESTA EM `vias_processadas`.
- `ANO`: DEIXE `None` PARA SOMAR TODOS OS ANOS JA ASSOCIADOS. UM ANO SO DA UM GRADIENTE
  MAIS FRACO, PORQUE A MAIORIA DAS VIAS FICA COM UM OU DOIS ACIDENTES.
- `LARGURA_MINIMA` E `LARGURA_MAXIMA`: ESPESSURA DA LINHA EM PIXELS, DA VIA COM MENOS
  ACIDENTES A COM MAIS. A ESPESSURA ANDA JUNTO COM A COR, PORQUE O OLHO PEGA ESPESSURA
  MAIS RAPIDO QUE MATIZ.
- `OPACIDADE_MINIMA`: TRANSPARENCIA DA VIA COM MENOS ACIDENTES. A COM MAIS E SEMPRE
  OPACA. E O QUE FAZ AS CENTENAS DE RUAS COM UM ACIDENTE RECUAREM PARA O FUNDO EM VEZ DE
  COMPETIREM COM AS PIORES.
- `MOSTRAR_VIAS_SEM_ACIDENTE`: LIGA A CAMADA CINZA COM O RESTO DA MALHA. VEM DESLIGADA
  NO MAPA E PODE SER LIGADA NO CONTROLE DE CAMADAS, NO CANTO.

In [25]:
# JSON LE A LISTA DE BAIRROS, QUE ESTA GRAVADA COMO TEXTO NO BANCO.
import json

# SQLITE3 E O BANCO LOCAL DO PROJETO.
import sqlite3

# SYS E USADO PARA GARANTIR QUE O PYTHON ENCONTRE O ARQUIVO config.py.
import sys

# DATETIME REGISTRA QUANDO O MAPA FOI GERADO.
from datetime import datetime, timezone

# PATH AJUDA A TRABALHAR COM CAMINHOS DE ARQUIVO.
from pathlib import Path

# NUMPY FAZ AS CONTAS DA ESCALA LOGARITMICA.
import numpy as np

# PANDAS FAZ A CONTAGEM E A JUNCAO COM AS VIAS.
import pandas as pd

# GEOPANDAS SEGURA A GEOMETRIA; SHAPELY LE O TEXTO WKT DO BANCO.
import geopandas as gpd
from shapely import wkt

# FOLIUM DESENHA O MAPA; BRANCA FAZ A ESCALA DE COR E A LEGENDA.
import branca.colormap as cm
import folium

# GARANTE QUE A PASTA projeto_v2 ESTEJA NO CAMINHO DE IMPORTACAO,
# INDEPENDENTE DE ONDE O JUPYTER FOI ABERTO.
for _candidata in (Path.cwd(), Path.cwd() / "projeto_v2", Path.cwd().parent):
    if (_candidata / "config.py").exists():
        sys.path.insert(0, str(_candidata))
        break

# CONFIGURACAO COMPARTILHADA: CAMINHOS, BANCO E NOMES DE COLUNA.
import config

# O QUE SERA PROCESSADO: MUNICIPIO, ANO E AS DEMAIS ESCOLHAS. TODAS ELAS VIVEM NO
# parametros.py -- E O UNICO ARQUIVO QUE VOCE EDITA ANTES DE RODAR.
import parametros

# =========================
# PARAMETROS DESTA EXECUCAO
# =========================

# MUNICIPIO DO MAPA. E O MESMO QUE O NOTEBOOK 02 USOU PARA BAIXAR A MALHA, PORQUE
# OS DOIS LEEM DO parametros.py.
CODIGO_IBGE = parametros.CODIGO_IBGE

# O MAPA USA ANO_MAPA, E NAO ANO: O 03 E O 04 PROCESSAM UM ANO DE CADA VEZ, ENQUANTO
# O MAPA COSTUMA QUERER TUDO O QUE JA FOI ASSOCIADO. `None` SOMA TODOS OS ANOS.
ANO = parametros.ANO_MAPA

# =========================
# APARENCIA DO MAPA
# =========================

# EXPOENTE DA ESCALA DE COR. 0.5 E A RAIZ QUADRADA, QUE E O EQUILIBRIO ESCOLHIDO
# NA ETAPA 3: SEPARA BEM AS PIORES VIAS SEM APAGAR O PELOTAO DO MEIO.
# MAIS PERTO DE 1.0 SEPARA MAIS O TOPO E APAGA MAIS O MEIO; MAIS PERTO DE 0.2 FAZ
# O CONTRARIO. NAO USE 0.
EXPOENTE_ESCALA = parametros.EXPOENTE_ESCALA

# ESPESSURA DA LINHA, EM PIXELS, DA VIA COM MENOS ACIDENTES A COM MAIS.
LARGURA_MINIMA = parametros.LARGURA_MINIMA
LARGURA_MAXIMA = parametros.LARGURA_MAXIMA

# TRANSPARENCIA DA VIA COM MENOS ACIDENTES. A COM MAIS E SEMPRE OPACA.
OPACIDADE_MINIMA = parametros.OPACIDADE_MINIMA

# LIGA A CAMADA CINZA COM AS VIAS QUE NAO RECEBERAM NENHUM ACIDENTE.
MOSTRAR_VIAS_SEM_ACIDENTE = parametros.MOSTRAR_VIAS_SEM_ACIDENTE

# ONDE O HTML SERA GRAVADO.
PASTA_MAPAS = config.PASTA_DADOS / "mapas"
ARQUIVO_SAIDA = PASTA_MAPAS / parametros.NOME_ARQUIVO_MAPA

# MOSTRA A CONFIGURACAO ATIVA PARA CONFERENCIA.
config.resumo()
print()
parametros.resumo()
print(f"SAIDA DO MAPA     : {ARQUIVO_SAIDA}")

BANCO DO V2       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\db\db_main.db
  EXISTE?         : SIM
PASTA TEMP        : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\temp
PASTA CACHE       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\cache
CHAVE DA IA       : DEFINIDA
MODELO DA IA      : deepseek-v4-flash
URL DA IA         : https://api.deepseek.com
EMBEDDINGS        : LIGADOS
MODELO EMBEDDINGS : intfloat/multilingual-e5-small

MUNICIPIO : 5201405
ANO       : TODOS OS JA ASSOCIADOS
SAIDA     : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\mapas\mapa_calor_vias.html


## 2. CONTAR OS ACIDENTES DE CADA VIA

A CONTAGEM E UM `GROUP BY` SOBRE `acidentes_revisado`, CONTANDO SO AS LINHAS COM
`assoc_status = 'ok'` -- OU SEJA, SO AS QUE O NOTEBOOK 04 CONSEGUIU VINCULAR A UMA VIA.

AS LINHAS `sem_via` FICAM DE FORA POR DEFINICAO: NAO HA VIA PARA COLORIR. ELAS APARECEM
NO RESUMO IMPRESSO, PARA VOCE SABER QUANTO DO SEU DADO NAO ENTROU NO MAPA.

### O AVISO DE VINCULO ORFAO

DEPOIS DE CONTAR, CONFERIMOS SE CADA `via_id_associada` AINDA EXISTE EM `vias_processadas`.

QUANDO NAO EXISTE, O VINCULO E **ORFAO**: FOI FEITO CONTRA UMA VERSAO ANTERIOR DA MALHA.
ISSO ACONTECE SEMPRE QUE O NOTEBOOK 02 E REEXECUTADO, PORQUE O `via_id` VEM DA GEOMETRIA
E QUALQUER EDICAO NO OPENSTREETMAP MOVE O CENTROIDE.

NAO HA CHAVE ESTRANGEIRA NO BANCO E NADA MAIS AVISA. SEM ESTA CONFERENCIA, O MAPA
SIMPLESMENTE MOSTRARIA MENOS ACIDENTES, SEM DIZER QUE ESTAVA ESCONDENDO ALGO.

### A TABELA `acidentes_por_via`

A CONTAGEM E GRAVADA NO BANCO. ASSIM O MAPA, O DASHBOARD E QUALQUER CONSULTA FUTURA LEEM
O MESMO NUMERO, EM VEZ DE CADA UM RECALCULAR O SEU.

In [26]:
def agora():
    """HORARIO ATUAL EM TEXTO, PARA REGISTRAR QUANDO O MAPA FOI GERADO."""
    return datetime.now(timezone.utc).isoformat()


def corrigir_texto(valor):
    """DESFAZ O MOJIBAKE DE TEXTO UTF-8 QUE FOI LIDO COMO LATIN-1.

    O NOTEBOOK 02 GRAVA NOMES COMO '10Aª Avenida' NO LUGAR DE '10ª Avenida'. O CONSERTO
    DE VERDADE E NA ORIGEM, NO 02 -- PORQUE O via_id E DERIVADO DO NOME, ENTAO HOJE ELE
    ESTA DERIVADO DE UM NOME ERRADO. AQUI SO ARRUMAMOS O QUE APARECE NA TELA.
    """
    if not isinstance(valor, str):
        return valor
    try:
        return valor.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return valor


def formatar_bairros(valor):
    """A COLUNA bairros E UM JSON EM TEXTO. VIRA UMA LISTA LEGIVEL PARA O TOOLTIP."""
    try:
        nomes = json.loads(valor) if valor else []
    except (TypeError, ValueError):
        return "-"

    if not isinstance(nomes, list) or not nomes:
        return "-"

    return ", ".join(corrigir_texto(n) for n in nomes)


conn = sqlite3.connect(str(config.BANCO))

# ---------------------------------------------------------------------------
# CONFERE AS DUAS DEPENDENCIAS ANTES DE QUALQUER CONSULTA.
# ---------------------------------------------------------------------------
tabelas = {l[0] for l in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}

if "vias_processadas" not in tabelas:
    raise RuntimeError(
        "A TABELA 'vias_processadas' NAO EXISTE NO BANCO.\n"
        "RODE O NOTEBOOK 02_malha_viaria.ipynb PRIMEIRO."
    )

if "acidentes_revisado" not in tabelas:
    raise RuntimeError(
        "A TABELA 'acidentes_revisado' NAO EXISTE NO BANCO.\n"
        "RODE OS NOTEBOOKS 03_revisao_enderecos.ipynb E 04_associacao_vias.ipynb PRIMEIRO."
    )

# ---------------------------------------------------------------------------
# CONFERE QUE A MALHA E DO MUNICIPIO QUE ESTAMOS MAPEANDO.
# ---------------------------------------------------------------------------
cidades = conn.execute(
    "SELECT cidade, estado, count(*) FROM vias_processadas GROUP BY 1, 2"
).fetchall()

print("MALHA EM vias_processadas:")
for cidade, estado, quantidade in cidades:
    print(f"   {corrigir_texto(cidade)} / {corrigir_texto(estado)}: {quantidade} VIAS")
print()
print("CONFIRA A OLHO QUE A CIDADE ACIMA E A DO CODIGO_IBGE DA ETAPA 1.")
print("A TABELA DE VIAS NAO GUARDA O CODIGO DO IBGE, ENTAO ISSO NAO DA PARA CONFERIR")
print("AUTOMATICAMENTE -- E ASSOCIAR ACIDENTE A RUA DE OUTRA CIDADE SAI MARCADO 'ok'.")
print()

# ---------------------------------------------------------------------------
# A CONTAGEM.
# ---------------------------------------------------------------------------
filtro_ano = ""
parametros = [str(CODIGO_IBGE)]

if ANO is not None:
    filtro_ano = f" AND {config.COLUNA_ANO} = ?"
    parametros.append(str(ANO))

contagem = pd.read_sql_query(
    f"SELECT via_id_associada AS via_id, count(*) AS acidentes "
    f"FROM acidentes_revisado "
    f"WHERE {config.COLUNA_MUNICIPIO} = ?{filtro_ano} "
    f"  AND assoc_status = 'ok' AND via_id_associada IS NOT NULL "
    f"GROUP BY via_id_associada",
    conn, params=parametros,
)

# O PANORAMA DO RECORTE, PARA SABER O QUE FICOU FORA DO MAPA.
panorama = pd.read_sql_query(
    f"SELECT COALESCE(assoc_status, '(nao processado)') AS status, count(*) AS linhas "
    f"FROM acidentes_revisado "
    f"WHERE {config.COLUNA_MUNICIPIO} = ?{filtro_ano} "
    f"GROUP BY 1 ORDER BY linhas DESC",
    conn, params=parametros,
)

print("SITUACAO DAS LINHAS DESTE RECORTE:")
print(panorama.to_string(index=False))
print()

if contagem.empty:
    raise RuntimeError(
        "NENHUM ACIDENTE ASSOCIADO NESTE RECORTE.\n"
        "RODE O NOTEBOOK 04_associacao_vias.ipynb PARA ESTE MUNICIPIO, OU CONFIRA O "
        "CODIGO_IBGE E O ANO DA ETAPA 1."
    )

# ---------------------------------------------------------------------------
# AS VIAS, COM GEOMETRIA.
# ---------------------------------------------------------------------------
vias = pd.read_sql_query(
    "SELECT via_id, nome_via, bairros, geometria FROM vias_processadas", conn
)

# ---------------------------------------------------------------------------
# O AVISO DE VINCULO ORFAO.
# ---------------------------------------------------------------------------
ids_existentes = set(vias["via_id"])
orfaos = contagem[~contagem["via_id"].isin(ids_existentes)]

acidentes_no_recorte = int(contagem["acidentes"].sum())
acidentes_orfaos = int(orfaos["acidentes"].sum())

print(f"ACIDENTES ASSOCIADOS      : {acidentes_no_recorte}")
print(f"VIAS DISTINTAS ATINGIDAS  : {len(contagem)}")
print(f"VINCULOS ORFAOS           : {len(orfaos)} VIAS, {acidentes_orfaos} ACIDENTES")

if acidentes_orfaos:
    fracao = acidentes_orfaos / acidentes_no_recorte * 100
    print()
    print("=" * 70)
    print(f"ATENCAO: {fracao:.1f}% DOS ACIDENTES APONTAM PARA UMA VIA QUE NAO EXISTE MAIS.")
    print("A MALHA FOI REPROCESSADA DEPOIS DA ASSOCIACAO. ESSES ACIDENTES NAO VAO")
    print("APARECER NO MAPA. RODE O NOTEBOOK 04 DE NOVO PARA REFAZER OS VINCULOS.")
    print("=" * 70)

# ---------------------------------------------------------------------------
# JUNCAO E GRAVACAO DA TABELA.
# ---------------------------------------------------------------------------
dados = contagem.merge(vias, on="via_id", how="inner")

tabela = dados[["via_id", "nome_via", "acidentes"]].copy()
tabela["codigo_ibge"] = str(CODIGO_IBGE)
tabela["ano"] = str(ANO) if ANO is not None else "todos"
tabela["gerado_em"] = agora()
tabela.to_sql("acidentes_por_via", conn, if_exists="replace", index=False)
conn.commit()

print()
print(f"TABELA 'acidentes_por_via' GRAVADA: {len(tabela)} LINHAS.")
print()
print("AS 15 VIAS COM MAIS ACIDENTES:")
maiores = dados.nlargest(15, "acidentes")[["nome_via", "acidentes"]].copy()
maiores["nome_via"] = maiores["nome_via"].apply(corrigir_texto)
print(maiores.to_string(index=False))

conn.close()

MALHA EM vias_processadas:
   Aparecida de Goiânia / Goias: 4845 VIAS

CONFIRA A OLHO QUE A CIDADE ACIMA E A DO CODIGO_IBGE DA ETAPA 1.
A TABELA DE VIAS NAO GUARDA O CODIGO DO IBGE, ENTAO ISSO NAO DA PARA CONFERIR
AUTOMATICAMENTE -- E ASSOCIAR ACIDENTE A RUA DE OUTRA CIDADE SAI MARCADO 'ok'.

SITUACAO DAS LINHAS DESTE RECORTE:
 status  linhas
     ok    6008
sem_via    2857

ACIDENTES ASSOCIADOS      : 6008
VIAS DISTINTAS ATINGIDAS  : 1387
VINCULOS ORFAOS           : 0 VIAS, 0 ACIDENTES

TABELA 'acidentes_por_via' GRAVADA: 1387 LINHAS.

AS 15 VIAS COM MAIS ACIDENTES:
                  nome_via  acidentes
         Avenida Rio Verde        288
     Avenida Independência        180
         Avenida São Paulo        166
            Rodovia GO-040        135
         Avenida Liberdade        120
               Avenida V-8        107
        Avenida Bela Vista         84
          Avenida São João         77
            Rodovia GO-040         64
            Avenida Brasil         61
        

## 3. A ESCALA DE COR

**ESTA ETAPA E A QUE DECIDE SE O MAPA SERVE PARA ALGUMA COISA.** ELA TAMBEM E A QUE MAIS
DEU TRABALHO, E VALE REGISTRAR O QUE NAO FUNCIONOU -- SAO QUATRO ARMADILHAS DIFERENTES.

ACIDENTE POR VIA E UMA DISTRIBUICAO MUITO TORTA. EM APARECIDA, DE 1387 VIAS ATINGIDAS,
QUASE METADE TEM **UM** ACIDENTE, E A PIOR TEM **288**.

### O QUE NAO FUNCIONOU

**FAIXAS POR QUANTIL UNIFORME.** COM SEIS FAIXAS, UM SEXTO DE TODAS AS VIAS CAI NA FAIXA
MAIS QUENTE POR CONSTRUCAO. CENTENAS DE RUAS VERMELHAS NAO APONTAM PONTO CRITICO NENHUM.

**FAIXAS CONCENTRADAS NA CAUDA.** RESOLVEU A QUANTIDADE E CRIOU OUTRO PROBLEMA: A FAIXA
MAIS QUENTE PASSOU A COBRIR DE 17 A 105 ACIDENTES. A PIOR VIA DA CIDADE E UMA RUA SEIS
VEZES MELHOR SAIAM COM O MESMO VERMELHO EXATO.

**QUALQUER ESQUEMA DE FAIXAS, NA VERDADE.** A ULTIMA FAIXA E SEMPRE ABERTA, ENTAO TUDO
QUE CAI NELA FICA IGUAL. A SAIDA E NAO TER FAIXA: COR CONTINUA.

**COR CONTINUA SOBRE O LOGARITMO.** PARECE A ESCOLHA OBVIA PARA DISTRIBUICAO TORTA, E E A
ERRADA AQUI. O LOGARITMO ESPALHA O **RODAPE** E COMPRIME A **PONTA** -- EXATAMENTE O
CONTRARIO DO QUE ESTE MAPA PRECISA. MEDIDO NOS DADOS REAIS, AS SEIS PIORES VIAS FICAVAM
DENTRO DE 17 PONTOS DE PALETA: 288 ACIDENTES EM 100%, 180 EM 92%, 166 EM 90%. TUDO
VERMELHO OUTRA VEZ.

### O QUE FUNCIONA: RAIZ QUADRADA

A ESCALA AQUI ELEVA A CONTAGEM A UM EXPOENTE ENTRE 0 E 1 -- POR PADRAO `0.5`, A RAIZ
QUADRADA. E O MEIO DO CAMINHO ENTRE LINEAR E LOGARITMICA, E OS TRES FORAM MEDIDOS NOS
DADOS REAIS ANTES DA ESCOLHA:

```text
                    SEPARACAO ENTRE     VIAS NA METADE
                    AS 6 PIORES         FRIA DA PALETA
    LINEAR          63 pontos           99,8%
    RAIZ QUADRADA   42 pontos           99,5%
    LOGARITMO       18 pontos           96,2%
```

LINEAR SEPARA O TOPO MELHOR QUE TUDO, MAS ESMAGA O MEIO: UMA VIA COM 20 ACIDENTES CAI EM
6,6% DA PALETA, AZUL IGUAL A DE UM ACIDENTE. A RAIZ QUADRADA COLOCA ESSA MESMA VIA EM
21,7% -- VIOLETA, VISIVEL -- E AINDA MANTEM 42 PONTOS ENTRE A PRIMEIRA E A SEXTA
COLOCADA.

**MEXA NO `EXPOENTE_ESCALA` SE QUISER OUTRO EQUILIBRIO.** MAIS PERTO DE `1.0` SEPARA O
TOPO E APAGA O MEIO; MAIS PERTO DE `0.2` FAZ O CONTRARIO.

DUAS VIAS COM 166 E 180 ACIDENTES CONTINUAM PARECIDAS, E ISSO ESTA CERTO: ELAS **SAO**
PARECIDAS, DIFEREM 8%. NENHUMA ESCALA HONESTA VAI PINTA-LAS DE CORES MUITO DIFERENTES.
PARA SABER QUAL E QUAL, PASSE O MOUSE -- O TOOLTIP DA O NUMERO EXATO.

A LEGENDA MOSTRA NUMERO DE ACIDENTE DE VERDADE, NAO O VALOR TRANSFORMADO. AS NOVE CORES
DA PALETA FICAM POSICIONADAS, NESTE RECORTE, EM 1, 9, 25, 49, 81, 121, 168, 224 E 288
ACIDENTES.

### A PALETA VAI DE AZUL APAGADO A VERMELHO PURO

DUAS PALETAS OBVIAS TAMBEM FORAM DESCARTADAS. A CLASSICA DE CALOR (`YlOrRd`) TEM TRES
VERMELHOS QUASE IDENTICOS NO TOPO: EM POLIGONO GRANDE ELES SE DISTINGUEM, EM **LINHA
FINA SOBRE FUNDO PRETO**, NAO. A `magma` DISTINGUE BEM, MAS TERMINA EM AMARELO QUASE
BRANCO -- AS PIORES VIAS SAIAM **CLARAS**, O CONTRARIO DO QUE SE ESPERA DE UM MAPA DE
RISCO.

A PALETA DAQUI VAI DE AZUL-CHUMBO APAGADO A VERMELHO PURO, PASSANDO POR VIOLETA, PURPURA
E CARMIM. ELA **NAO PASSA PELO AMARELO** DE PROPOSITO: AMARELO E A COR MAIS CLARA DE
QUALQUER PALETA DE CALOR, ENTAO SE FICASSE NO MEIO, O MEIO BRILHARIA MAIS QUE O TOPO.

In [27]:
# PALETA DO AZUL-CHUMBO APAGADO AO VERMELHO PURO. NAO PASSA PELO AMARELO DE
# PROPOSITO: AMARELO NO MEIO BRILHARIA MAIS QUE O TOPO. ASSIM O BRILHO E A
# SATURACAO CRESCEM ATE O FIM, E O FIM E VERMELHO -- QUE E O QUE SE ESPERA DE UM
# MAPA DE RISCO.
PALETA = ["#2b3f6b", "#3d5a9c", "#5c4b9e", "#7c3f96", "#9e3480",
          "#bf2e66", "#d92847", "#ef2130", "#ff1414"]

valores = dados["acidentes"].to_numpy()

minimo = int(valores.min())
maximo = int(valores.max())

# CASO DEGENERADO: TODAS AS VIAS COM A MESMA CONTAGEM. SEM ISSO A DIVISAO
# ESTOURARIA MAIS ABAIXO.
if maximo <= minimo:
    maximo = minimo + 1

# A CONTAGEM E ELEVADA A ESTE EXPOENTE ANTES DE VIRAR COR. VEJA A ETAPA 3:
# 1.0 SEPARA O TOPO E APAGA O MEIO, VALORES BAIXOS FAZEM O CONTRARIO.
base = minimo ** EXPOENTE_ESCALA
teto = maximo ** EXPOENTE_ESCALA


def fracao_de(quantidade):
    """POSICAO DE UMA CONTAGEM NA ESCALA, DE 0.0 (MENOS) A 1.0 (MAIS)."""
    posicao = (quantidade ** EXPOENTE_ESCALA - base) / (teto - base)

    # PRENDE ENTRE 0 E 1 PARA O CASO DEGENERADO ACIMA.
    return min(max(float(posicao), 0.0), 1.0)


# AS CORES DA PALETA SAO POSICIONADAS EM PASSOS IGUAIS DO VALOR TRANSFORMADO, E
# DEPOIS TRAZIDAS DE VOLTA PARA CONTAGEM. ASSIM O branca INTERPOLA NA ESCALA QUE
# QUEREMOS, MAS A LEGENDA CONTINUA MOSTRANDO NUMERO DE ACIDENTE DE VERDADE.
posicoes = [
    (base + i / (len(PALETA) - 1) * (teto - base)) ** (1 / EXPOENTE_ESCALA)
    for i in range(len(PALETA))
]

escala = cm.LinearColormap(
    colors=PALETA,
    index=posicoes,
    vmin=minimo,
    vmax=maximo,
    caption=f"ACIDENTES POR VIA (COR EM ESCALA DE POTENCIA {EXPOENTE_ESCALA})",
)


def largura_de(quantidade):
    """ESPESSURA DA LINHA, CRESCENDO JUNTO COM A COR."""
    return LARGURA_MINIMA + fracao_de(quantidade) * (LARGURA_MAXIMA - LARGURA_MINIMA)


def opacidade_de(quantidade):
    """OPACIDADE DA LINHA. A VIA COM MENOS ACIDENTES E A MAIS TRANSPARENTE.

    TRES SINAIS ANDAM JUNTOS -- COR, ESPESSURA E OPACIDADE -- PORQUE UM SO NAO
    SEPARA MIL LINHAS SOBREPOSTAS NUM MAPA DE CIDADE.
    """
    return OPACIDADE_MINIMA + fracao_de(quantidade) * (1.0 - OPACIDADE_MINIMA)


print(f"VIAS COM ACIDENTE : {len(dados)}")
print(f"MINIMO / MAXIMO   : {int(valores.min())} / {int(valores.max())} ACIDENTES")
print(f"EXPOENTE          : {EXPOENTE_ESCALA}")
print()
print("ONDE AS NOVE CORES DA PALETA CAEM, EM ACIDENTES:")
print("   " + ", ".join(f"{round(p)}" for p in posicoes))
print()
print("AS 12 VIAS COM MAIS ACIDENTES E A COR DE CADA UMA:")
for _, linha in dados.nlargest(12, "acidentes").iterrows():
    quantidade = int(linha["acidentes"])
    print(f"   {quantidade:>5}  {escala(quantidade)}  "
          f"POSICAO {fracao_de(quantidade) * 100:>5.1f}%  "
          f"LARGURA {largura_de(quantidade):.1f}  "
          f"{corrigir_texto(linha['nome_via'])}")

print()
print("E ONDE CAI O PELOTAO DE BAIXO, QUE PRECISA FICAR APAGADO:")
for referencia in (1, 2, 5, 10, 20, 50):
    if minimo <= referencia <= maximo:
        print(f"   {referencia:>5} ACIDENTES  ->  {escala(referencia)}  "
              f"POSICAO {fracao_de(referencia) * 100:>5.1f}%  "
              f"LARGURA {largura_de(referencia):.1f}")

VIAS COM ACIDENTE : 1387
MINIMO / MAXIMO   : 1 / 288 ACIDENTES

ONDE ALGUMAS CONTAGENS CAEM NA ESCALA:
       1 ACIDENTES  ->  #2b3f6bff  POSICAO   0.0%  LARGURA 1.5
       2 ACIDENTES  ->  #3c599bff  POSICAO  12.2%  LARGURA 2.4
       5 ACIDENTES  ->  #63489cff  POSICAO  28.4%  LARGURA 3.6
      10 ACIDENTES  ->  #823d92ff  POSICAO  40.7%  LARGURA 4.5
      20 ACIDENTES  ->  #a4337bff  POSICAO  52.9%  LARGURA 5.5
      50 ACIDENTES  ->  #cb2b58ff  POSICAO  69.1%  LARGURA 6.7
     288 ACIDENTES  ->  #ff1414ff  POSICAO 100.0%  LARGURA 9.0

AS 12 VIAS COM MAIS ACIDENTES E A COR DE CADA UMA:
     288  #ff1414ff  POSICAO 100.0%  Avenida Rio Verde
     180  #f41d28ff  POSICAO  91.7%  Avenida Independência
     166  #f21e2bff  POSICAO  90.3%  Avenida São Paulo
     135  #ed2132ff  POSICAO  86.6%  Rodovia GO-040
     120  #e92337ff  POSICAO  84.5%  Avenida Liberdade
     107  #e5243bff  POSICAO  82.5%  Avenida V-8
      84  #de2642ff  POSICAO  78.2%  Avenida Bela Vista
      77  #dc2745ff  PO

## 4. O MAPA

TRES DECISOES DE CONSTRUCAO, TODAS COM MOTIVO PRATICO:

**UMA CAMADA `GeoJson`, NAO UMA `PolyLine` POR VIA.** A MALHA TEM MILHARES DE VIAS. UM
OBJETO DE MAPA PARA CADA UMA GERA UM HTML DE DEZENAS DE MEGABYTES QUE TRAVA O NAVEGADOR.
UMA CAMADA UNICA COM `style_function` DESENHA O MESMO DESENHO EM UMA FRACAO DO TAMANHO.

**FUNDO ESCURO.** O `CartoDB dark_matter` FAZ O AMARELO E O VERMELHO SALTAREM. NUM FUNDO
CLARO, A FAIXA MAIS FRIA DA ESCALA PRATICAMENTE DESAPARECE.

**A MALHA SEM ACIDENTE FICA EM CAMADA SEPARADA.** ELA DA CONTEXTO -- MOSTRA ONDE A
CIDADE TEM RUA E NAO TEM ACIDENTE REGISTRADO -- MAS PESA E POLUI. VEM DESLIGADA, E O
CONTROLE NO CANTO SUPERIOR DIREITO LIGA.

O TOOLTIP DE CADA VIA MOSTRA O NOME, OS BAIRROS POR ONDE ELA PASSA E A CONTAGEM.

**O HTML PRECISA DE INTERNET PARA ABRIR**: O FOLIUM BUSCA O LEAFLET E OS AZULEJOS DO
MAPA NA REDE. O ARQUIVO NAO CARREGA O MAPA-BASE OFFLINE.

In [28]:
# ---------------------------------------------------------------------------
# GEOMETRIA: O BANCO GUARDA WKT EM TEXTO, EM WGS84 (EPSG:4326).
# ---------------------------------------------------------------------------
dados_mapa = dados.copy()
dados_mapa["nome_via"] = dados_mapa["nome_via"].apply(corrigir_texto)
dados_mapa["bairros_texto"] = dados_mapa["bairros"].apply(formatar_bairros)

gdf = gpd.GeoDataFrame(
    dados_mapa,
    geometry=dados_mapa["geometria"].apply(wkt.loads),
    crs="EPSG:4326",
)

# DESCARTA GEOMETRIA VAZIA OU INVALIDA, QUE O FOLIUM NAO CONSEGUE DESENHAR.
antes = len(gdf)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
if len(gdf) < antes:
    print(f"AVISO: {antes - len(gdf)} VIAS DESCARTADAS POR GEOMETRIA VAZIA OU INVALIDA.")

# DESENHA AS PIORES POR CIMA: NUM CRUZAMENTO, QUEM FICA VISIVEL E A VIA DE MAIS
# ACIDENTES, E NAO O QUE POR ACASO VEIO DEPOIS NA TABELA.
gdf = gdf.sort_values("acidentes")

# ---------------------------------------------------------------------------
# O MAPA, CENTRADO NA PROPRIA MALHA.
# ---------------------------------------------------------------------------
minx, miny, maxx, maxy = gdf.total_bounds
mapa = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    tiles="CartoDB dark_matter",
)

# ENQUADRA O MAPA NA MALHA, EM VEZ DE CONFIAR NO ZOOM FIXO.
mapa.fit_bounds([[miny, minx], [maxy, maxx]])


def estilo(feature):
    """COR, ESPESSURA E OPACIDADE DE UMA VIA, TODAS A PARTIR DA CONTAGEM.

    NAO HA FAIXA NENHUMA AQUI: OS TRES SINAIS SAO CONTINUOS NO LOGARITMO DA
    CONTAGEM. E POR ISSO QUE A VIA DE 105 ACIDENTES E A DE 17 SAEM DIFERENTES.
    """
    quantidade = feature["properties"]["acidentes"]
    return {
        "color": escala(quantidade),
        "weight": largura_de(quantidade),
        "opacity": opacidade_de(quantidade),
    }


COLUNAS_MAPA = ["nome_via", "bairros_texto", "acidentes", "geometry"]

folium.GeoJson(
    gdf[COLUNAS_MAPA],
    style_function=estilo,
    # O REALCE SO MUDA A ESPESSURA: MUDAR A COR ATRAPALHARIA A LEITURA DA ESCALA.
    highlight_function=lambda 
    feature: {"weight": LARGURA_MAXIMA + 3, "opacity": 1.0},
    tooltip=folium.GeoJsonTooltip(
        fields=["nome_via", "bairros_texto", "acidentes"],
        aliases=["VIA:", "BAIRROS:", "ACIDENTES:"],
        sticky=True,
    ),
    name="ACIDENTES POR VIA",
).add_to(mapa)

# ---------------------------------------------------------------------------
# CAMADA DE CONTEXTO: O RESTO DA MALHA, EM CINZA, DESLIGADA POR PADRAO.
# ---------------------------------------------------------------------------
if MOSTRAR_VIAS_SEM_ACIDENTE:
    sem_acidente = vias[~vias["via_id"].isin(set(dados["via_id"]))].copy()

    if not sem_acidente.empty:
        gdf_cinza = gpd.GeoDataFrame(
            sem_acidente,
            geometry=sem_acidente["geometria"].apply(wkt.loads),
            crs="EPSG:4326",
        )
        gdf_cinza = gdf_cinza[gdf_cinza.geometry.notna() & ~gdf_cinza.geometry.is_empty]

        folium.GeoJson(
            gdf_cinza[["geometry"]],
            # BEM APAGADA: E CONTEXTO, NAO DADO. NAO PODE COMPETIR COM O DADO.
            style_function=lambda feature: {"color": "#333333", "weight": 0.8,
                                            "opacity": 0.35},
            name=f"MALHA SEM ACIDENTE ({len(gdf_cinza)} VIAS)",
            show=False,
        ).add_to(mapa)

        print(f"CAMADA DE CONTEXTO: {len(gdf_cinza)} VIAS SEM ACIDENTE.")

# A LEGENDA DA ESCALA E O CONTROLE DE CAMADAS.
escala.add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)

# ---------------------------------------------------------------------------
# GRAVACAO.
# ---------------------------------------------------------------------------
PASTA_MAPAS.mkdir(parents=True, exist_ok=True)
mapa.save(str(ARQUIVO_SAIDA))

tamanho_mb = ARQUIVO_SAIDA.stat().st_size / (1024 * 1024)
print()
print(f"MAPA GRAVADO EM : {ARQUIVO_SAIDA}")
print(f"TAMANHO         : {tamanho_mb:.1f} MB")
print(f"VIAS DESENHADAS : {len(gdf)}")
print()
print("ABRA O ARQUIVO NO NAVEGADOR. PRECISA DE INTERNET PARA CARREGAR O MAPA-BASE.")

# MOSTRA O MAPA AQUI DENTRO DO NOTEBOOK TAMBEM.
mapa


MAPA GRAVADO EM : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\mapas\mapa_calor_vias.html
TAMANHO         : 1.0 MB
VIAS DESENHADAS : 1387

ABRA O ARQUIVO NO NAVEGADOR. PRECISA DE INTERNET PARA CARREGAR O MAPA-BASE.


## O QUE OLHAR NO MAPA, E COM QUE DESCONFIANCA

**AS VIAS VERMELHAS SAO AS MAIS MOVIMENTADAS, NAO NECESSARIAMENTE AS MAIS PERIGOSAS.**
UMA AVENIDA DE 8 KM COM 200 ACIDENTES E UMA RUA DE 300 M COM 200 ACIDENTES SAO PROBLEMAS
MUITO DIFERENTES. A TABELA `vias_processadas` TEM `comp_usado`, O COMPRIMENTO DA VIA --
DIVIDIR A CONTAGEM POR ELE DA ACIDENTE POR QUILOMETRO, QUE E A MEDIDA QUE APONTA TRECHO
PERIGOSO. ISSO **NAO** ESTA NESTE MAPA.

**O NOME REPETIDO CONTAMINA A CONTAGEM.** A ETAPA 9 DO NOTEBOOK 04 MEDE QUANTO DO SEU
RESULTADO FOI ASSOCIADO A UMA VIA CUJO NOME EXISTE VARIAS VEZES NA CIDADE. NESSES CASOS
A ESCOLHA ENTRE AS HOMONIMAS PODE TER SIDO ARBITRARIA, E ENTAO **A CONTAGEM FOI PARA A
VIA ERRADA** -- UMA FICA VERMELHA SEM MERECER E A OUTRA FICA FRIA SEM MERECER. OLHE
AQUELE PERCENTUAL ANTES DE APRESENTAR ESTE MAPA A ALGUEM.

**A COR E RANKING, NAO VALOR.** POR CONSTRUCAO, MAIS OU MENOS UM SETIMO DAS VIAS CAI NA
FAIXA MAIS QUENTE. SE A CIDADE INTEIRA MELHORAR PELA METADE, O MAPA FICA IGUAL. PARA
COMPARAR ANOS, USE OS NUMEROS DA TABELA `acidentes_por_via`, NAO AS CORES.